# 04 — The protein layer

**The payoff notebook.** Does the transcriptional story hold at protein level?

The four quadrants of an RNA-vs-protein log2FC comparison:

| | protein down | protein flat |
|---|---|---|
| **RNA down** | concordant loss of function | buffering, protein half-life, or ADT floor |
| **RNA flat** | **post-transcriptional regulation** | no effect |

The bottom-left quadrant is the mechanistically interesting one, and it is
invisible to any RNA-only screen.

**Positive control:** the published mechanism is that CD58 protein is *not*
IFN-γ-inducible while MHC-I is, and that CD58 loss confers immune evasion
without compromising MHC. Recovering that independently validates the whole
pipeline.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig

cfg = load_config()
panels = load_panels()
P = paths(cfg)
SEED = set_seed(cfg)
apply_style(cfg)

sc.settings.verbosity = 1
print(f"repo: {P.root}")
print(f"seed: {SEED}")


In [ ]:
import mudata as md
mdata = md.read(P.data_interim / "frangieh_qc.h5mu")
rna, adt = mdata["rna"], mdata["adt"]
s = cfg["schema"]["obs"]
print("ADT panel:", list(adt.var_names))

## 1. CLR normalisation

Per cell, margin 0 — the standard for ADT, since library size differences between cells dominate raw counts.

In [ ]:
import muon as mu

adt.layers["counts"] = adt.X.copy()
mu.prot.pp.clr(adt, axis=cfg["protein"]["clr_margin"])
adt

## 2. Perturbation effects in ADT space

Same condition-matched-control logic as nb02. With only ~20 features, a per-cell rank test is fine; no pseudobulk needed.

In [ ]:
# TODO: per (perturbation, condition), CLR mean difference vs matched control
# -> adt_effects DataFrame [perturbation, condition, adt_feature, lfc, pvalue]
print("[stub] ADT effect table")

## 3. RNA vs protein

**Handle many-to-one mappings explicitly.** An MHC-I antibody detecting
HLA-A/B/C cannot be naively correlated against a single transcript — that
mismatch would masquerade as discordance. Use `panels.yaml -> adt.adt_to_rna`,
which you populated in nb01.

In [ ]:
adt_to_rna = panels["adt"]["adt_to_rna"]
if not adt_to_rna:
    raise ValueError(
        "panels.yaml -> adt.adt_to_rna is still the empty stub. "
        "Populate it from config/panels_observed.yaml (written by nb01) before "
        "running this notebook."
    )
adt_to_rna

In [ ]:
# TODO: join adt_effects to the RNA signature matrix on (perturbation,
# condition, gene), aggregating multi-gene ADT features sensibly, then produce
# the quadrant scatter.
print("[stub] quadrant analysis")

## 4. Markers of interest

Foreground these regardless of where the screen ranks them.

In [ ]:
markers = panels["adt"]["markers_of_interest"]
print(markers)

# TODO: for each marker, plot CLR level by condition in control cells
# (is it IFN-gamma inducible?) and by perturbation (what knocks it down?).
#
# The CD58 result to look for: flat across conditions in controls -- i.e. NOT
# IFN-gamma induced -- while MHC-I and PD-L1 rise sharply. That is the
# published mechanism, recovered independently.
print("[stub] marker panels")